In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")

print("Path to dataset files:", path)

100%|██████████| 230M/230M [04:51<00:00, 826kB/s]  

Extracting files...


Path to dataset files: C:\Users\Manu\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1


In [3]:
import shutil

from pathlib import Path

destination = Path("data/raw/network-intrusion-dataset")
if  destination.exists():
    shutil.rmtree(destination)

shutil.copytree(path, destination)

WindowsPath('data/raw/network-intrusion-dataset')

#DATASET FILES

In [30]:
import os
import pandas as pd
import numpy as np



PROJECT_ROOT = "D:\\Development\\Python Projects\\network-traffic-anomaly-detection"

os.chdir(PROJECT_ROOT)
data_dir = Path("data/raw/network-intrusion-dataset/versions/1")

summary = []

csv_files = sorted(data_dir.glob("*.csv"))

for file in csv_files:
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()

    summary.append(
        {
            "file_name": file.name,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "label_column_exists" : "Label" in df.columns,
            "num_missing_vales" : df.isna().sum().sum(),
            "num_duplicate_rows" : df.duplicated().sum(),
        }
    )

summary_df = pd.DataFrame(summary).sort_values("file_name").reset_index(drop=True)

summary_df.style.set_properties(**{
    "text-align": "left"
}).set_table_styles([
    {"selector": "th", "props": [("text-align", "left")]}
])




,file_name,rows,cols,label_column_exists,num_missing_vales,num_duplicate_rows
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,79,True,4,2633
1,Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,286467,79,True,15,72353
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,79,True,28,6888
3,Monday-WorkingHours.pcap_ISCX.csv,529918,79,True,64,26935
4,Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,288602,79,True,18,35630
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,170366,79,True,20,6066
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,79,True,201,24065
7,Wednesday-workingHours.pcap_ISCX.csv,692703,79,True,1008,81909


In [31]:
for file in csv_files:
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()

    print("\n" + "="*90)
    print(file.name)
    print("="*90)
    print(df["Label"].value_counts())


Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64

Friday-WorkingHours-Morning.pcap_ISCX.csv
Label
BENIGN    189067
Bot         1966
Name: count, dtype: int64

Monday-WorkingHours.pcap_ISCX.csv
Label
BENIGN    529918
Name: count, dtype: int64

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Label
BENIGN          288566
Infiltration        36
Name: count, dtype: int64

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Label
BENIGN                        168186
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64

Tuesday-WorkingHours.pcap_ISCX.csv
Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

Wednesday-workingHours.pcap_ISCX.csv
Label
BENIGN        

##Baseline Comparision Setup

From the label distribution analysis, `Monday-WorkingHours.pcap_ISCX.csv` contains only BENIGN traffic. Therefore, it can be used as the normal baseline dataset.

For the first experiment, we compare this normal baseline against `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv`, which contains both BENIGN and DDoS traffic. This allows us to test whether an anomaly detection model trained on normal traffic can assign higher anomaly scores to DDoS flows.

The initial experiment is:

- Training baseline: Monday BENIGN traffic
- Evaluation file: Friday afternoon BENIGN + DDoS traffic
- Goal: learn normal network behavior from Monday and detect DDoS traffic as anomalous


In [32]:
baseline_comparison = pd.DataFrame({
    "role" : ["Training baseline", "Evaluation data"],
    "file_name" : [
        "Monday-WorkingHours.pcap_ISCX.csv",
        "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
    ],
    "traffic_type" : [
        "Only BENIGN traffic",
        "Mixed BENIGN and DDoS traffic"
    ],
    "purpose" : [
        "Learn normal network behavior",
        "Evaluate whether DDoS traffic is detected as anomalous"
    ]
})

baseline_comparison

,role,file_name,traffic_type,purpose
0,Training baseline,Monday-WorkingHours.pcap_ISCX.csv,Only BENIGN traffic,Learn normal network behavior
1,Evaluation data,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Mixed BENIGN and DDoS traffic,Evaluate whether DDoS traffic is detected as a...


In [33]:
train_file = data_dir / "Monday-WorkingHours.pcap_ISCX.csv"
test_file = data_dir / "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

print("Training baseline file:")
print(train_df["Label"].value_counts())

print("\nEvaluation file:")
print(test_df["Label"].value_counts())

Training baseline file:
Label
BENIGN    529918
Name: count, dtype: int64

Evaluation file:
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


The Monday file is selected as the baseline because it contains only BENIGN samples. This is suitable for unsupervised anomaly detection because the model can learn the structure of normal traffic without being exposed to attack samples during training.

The Friday afternoon DDoS file is selected as the first evaluation dataset because it contains both BENIGN and DDoS samples. The labels are not used during training, but they are useful for evaluating whether the anomaly detector can distinguish normal traffic from attack traffic.